In [ ]:
import re
from datasets import load_dataset
import numpy as np 
from collections import Counter
import gc

In [2]:

# Use 'strict_small' as identified in your error message
dataset = load_dataset(
    "cambridge-climb/BabyLM", 
    name="strict_small", 
    trust_remote_code=True
)


print(len(dataset['train']))
print(len(dataset['test']))

335858
342095


In [3]:
block_size = 8
embed_dim = 64
hidden_size = 512
batch_size = 32
text = " ".join(sample.lower() + " <eos>" for sample in dataset['train']['text'])
all_words = text.split()

#tokenize
words = text.split()
print(len(words))
#vocab


word_counts = Counter(words)
unique_words = sorted([w for w in set(words) if word_counts[w] >= 20])
unique_words.append("<unk>") # for rare words
vocab_size = len(unique_words)  # ~160k
print(vocab_size)

stoi = {w: i for i, w in enumerate(unique_words)}
itos = {i: w for i, w in enumerate(unique_words)}

# Encode with <unk> fallback
data = np.array([stoi.get(w, stoi["<unk>"]) for w in words])



X, y = [], []

for i in range(len(data) - block_size):
    context = data[i:i + block_size]         #[i,i+1, ...,i+7]
    targets = data[i + 1:i + block_size + 1] #[i+1,i+2, ...,i+8]
    X.append(context)
    y.append(targets)

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Train/test split (90/10)
split_idx = int(len(X) * 0.9)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")




9783451
24875
X shape: (9783443, 8)
y shape: (9783443, 8)
Train: 8,805,098 | Test: 978,345


In [ ]:
#Claude Code
import Metal
import numpy as np
# READY. NOT GOOD 
# ============================================
# GPU Setup and Pipeline Initialization
# ============================================

device = Metal.MTLCreateSystemDefaultDevice()
print(f"Metal works! GPU: {device.name()}")

# Load and compile shader
with open('kernel_claude_test.metal', 'r') as f:
    shader_source = f.read()

library, error = device.newLibraryWithSource_options_error_(
    shader_source, None, None
)

if library is None:
    print("❌ SHADER COMPILATION FAILED!")
    print(f"Error: {error}")
    raise SystemExit(1)

print("✓ Shader compiled!")

# Get all kernel functions
dot_fn = library.newFunctionWithName_("gpu_dot")
embedding_lookup_fn = library.newFunctionWithName_("gpu_embedding_lookup")
relu_fn = library.newFunctionWithName_("gpu_relu")
relu_backward_fn = library.newFunctionWithName_("gpu_relu_backward")
gelu_fn = library.newFunctionWithName_("gpu_gelu")
gelu_backward_fn = library.newFunctionWithName_("gpu_gelu_backward")
softmax_fn = library.newFunctionWithName_("gpu_softmax")
softmax_4d_fn = library.newFunctionWithName_("gpu_softmax_4d")
transpose_2d_fn = library.newFunctionWithName_("gpu_transpose_2d")
transpose_4d_fn = library.newFunctionWithName_("gpu_transpose_4d")
element_add_fn = library.newFunctionWithName_("gpu_element_add")
element_mult_fn = library.newFunctionWithName_("gpu_element_mult")
layer_norm_forward_fn = library.newFunctionWithName_("gpu_layer_norm_forward")
layer_norm_backward_dx_fn = library.newFunctionWithName_("gpu_layer_norm_backward_dx")
layer_norm_backward_gamma_beta_fn = library.newFunctionWithName_("gpu_layer_norm_backward_gamma_beta")
batched_matmul_fn = library.newFunctionWithName_("gpu_batched_matmul")
scale_fn = library.newFunctionWithName_("gpu_scale")
add_mask_fn = library.newFunctionWithName_("gpu_add_mask")

scalar_div_fn = library.newFunctionWithName_("gpu_scalar_div")
scalar_mult_fn = library.newFunctionWithName_("gpu_scalar_mult")
bias_add_3d_fn = library.newFunctionWithName_("gpu_bias_add_3d")
bias_add_2d_fn = library.newFunctionWithName_("gpu_bias_add_2d")
sum_3d_to_1d_fn = library.newFunctionWithName_("gpu_sum_3d_to_1d")
sum_2d_to_1d_fn = library.newFunctionWithName_("gpu_sum_2d_to_1d")
element_sub_fn = library.newFunctionWithName_("gpu_element_sub")
softmax_backward_4d_fn = library.newFunctionWithName_("gpu_softmax_backward_4d")
cross_entropy_fn = library.newFunctionWithName_("gpu_cross_entropy_loss")
softmax_ce_backward_fn = library.newFunctionWithName_("gpu_softmax_ce_backward")
scatter_add_fn = library.newFunctionWithName_("gpu_scatter_add")
adam_update_fn = library.newFunctionWithName_("gpu_adam_update")

# Create pipelines
pipeline_scalar_div = device.newComputePipelineStateWithFunction_error_(scalar_div_fn, None)[0]
pipeline_scalar_mult = device.newComputePipelineStateWithFunction_error_(scalar_mult_fn, None)[0]
pipeline_bias_add_3d = device.newComputePipelineStateWithFunction_error_(bias_add_3d_fn, None)[0]
pipeline_bias_add_2d = device.newComputePipelineStateWithFunction_error_(bias_add_2d_fn, None)[0]
pipeline_sum_3d_to_1d = device.newComputePipelineStateWithFunction_error_(sum_3d_to_1d_fn, None)[0]
pipeline_sum_2d_to_1d = device.newComputePipelineStateWithFunction_error_(sum_2d_to_1d_fn, None)[0]
pipeline_element_sub = device.newComputePipelineStateWithFunction_error_(element_sub_fn, None)[0]
pipeline_softmax_backward_4d = device.newComputePipelineStateWithFunction_error_(softmax_backward_4d_fn, None)[0]
pipeline_cross_entropy = device.newComputePipelineStateWithFunction_error_(cross_entropy_fn, None)[0]
pipeline_softmax_ce_backward = device.newComputePipelineStateWithFunction_error_(softmax_ce_backward_fn, None)[0]
pipeline_scatter_add = device.newComputePipelineStateWithFunction_error_(scatter_add_fn, None)[0]
pipeline_adam_update = device.newComputePipelineStateWithFunction_error_(adam_update_fn, None)[0]

# Create pipelines
pipeline_dot = device.newComputePipelineStateWithFunction_error_(dot_fn, None)[0]
pipeline_embedding = device.newComputePipelineStateWithFunction_error_(embedding_lookup_fn, None)[0]
pipeline_relu = device.newComputePipelineStateWithFunction_error_(relu_fn, None)[0]
pipeline_relu_backward = device.newComputePipelineStateWithFunction_error_(relu_backward_fn, None)[0]
pipeline_gelu = device.newComputePipelineStateWithFunction_error_(gelu_fn, None)[0]
pipeline_gelu_backward = device.newComputePipelineStateWithFunction_error_(gelu_backward_fn, None)[0]
pipeline_softmax = device.newComputePipelineStateWithFunction_error_(softmax_fn, None)[0]
pipeline_softmax_4d = device.newComputePipelineStateWithFunction_error_(softmax_4d_fn, None)[0]
pipeline_transpose_2d = device.newComputePipelineStateWithFunction_error_(transpose_2d_fn, None)[0]
pipeline_transpose_4d = device.newComputePipelineStateWithFunction_error_(transpose_4d_fn, None)[0]
pipeline_element_add = device.newComputePipelineStateWithFunction_error_(element_add_fn, None)[0]
pipeline_element_mult = device.newComputePipelineStateWithFunction_error_(element_mult_fn, None)[0]
pipeline_layer_norm_forward = device.newComputePipelineStateWithFunction_error_(layer_norm_forward_fn, None)[0]
pipeline_layer_norm_backward_dx = device.newComputePipelineStateWithFunction_error_(layer_norm_backward_dx_fn, None)[0]
pipeline_layer_norm_backward_gamma_beta = device.newComputePipelineStateWithFunction_error_(layer_norm_backward_gamma_beta_fn, None)[0]
pipeline_batched_matmul = device.newComputePipelineStateWithFunction_error_(batched_matmul_fn, None)[0]
pipeline_scale = device.newComputePipelineStateWithFunction_error_(scale_fn, None)[0]
pipeline_add_mask = device.newComputePipelineStateWithFunction_error_(add_mask_fn, None)[0]

command_queue = device.newCommandQueue()

print("✓ All pipelines created!")


# ============================================
# Helper Functions
# ============================================

def gpu_scalar_div(input_arr, scalar):
    """Divide all elements by scalar"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    total = input_arr.size
    shape = input_arr.shape
    
    in_buffer = create_buffer(input_arr.flatten())
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_scalar_div)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(float(scalar)), 0, 2)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, shape)


def gpu_scalar_mult(input_arr, scalar):
    """Multiply all elements by scalar"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    total = input_arr.size
    shape = input_arr.shape
    
    in_buffer = create_buffer(input_arr.flatten())
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_scalar_mult)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(float(scalar)), 0, 2)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, shape)


def gpu_bias_add_3d(input_arr, bias):
    """Add bias to 3D tensor: (batch, seq, dim) + (dim,)"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    bias = np.ascontiguousarray(bias, dtype=np.float32)
    
    batch, seq, dim = input_arr.shape
    
    in_buffer = create_buffer(input_arr)
    bias_buffer = create_buffer(bias)
    out_buffer = device.newBufferWithLength_options_(batch * seq * dim * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_bias_add_3d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(bias_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(batch), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(seq), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(dim), 0, 5)
    
    threads_per_grid = Metal.MTLSize(dim, seq, batch)
    threads_per_group = Metal.MTLSize(min(dim, 16), min(seq, 8), min(batch, 4))
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (batch, seq, dim))


def gpu_bias_add_2d(input_arr, bias):
    """Add bias to 2D tensor: (rows, cols) + (cols,)"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    bias = np.ascontiguousarray(bias, dtype=np.float32)
    
    rows, cols = input_arr.shape
    
    in_buffer = create_buffer(input_arr)
    bias_buffer = create_buffer(bias)
    out_buffer = device.newBufferWithLength_options_(rows * cols * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_bias_add_2d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(bias_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(rows), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(cols), 0, 4)
    
    threads_per_grid = Metal.MTLSize(cols, rows, 1)
    threads_per_group = Metal.MTLSize(min(cols, 16), min(rows, 16), 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (rows, cols))


def gpu_sum_3d_to_1d(input_arr):
    """Sum 3D tensor along axes 0 and 1: (batch, seq, dim) -> (dim,)"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    batch, seq, dim = input_arr.shape
    
    in_buffer = create_buffer(input_arr)
    out_buffer = device.newBufferWithLength_options_(dim * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_sum_3d_to_1d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(batch), 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(seq), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(dim), 0, 4)
    
    threads_per_grid = Metal.MTLSize(dim, 1, 1)
    threads_per_group = Metal.MTLSize(min(dim, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (dim,))


def gpu_sum_2d_to_1d(input_arr):
    """Sum 2D tensor along axis 0: (rows, cols) -> (cols,)"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    rows, cols = input_arr.shape
    
    in_buffer = create_buffer(input_arr)
    out_buffer = device.newBufferWithLength_options_(cols * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_sum_2d_to_1d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(rows), 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(cols), 0, 3)
    
    threads_per_grid = Metal.MTLSize(cols, 1, 1)
    threads_per_group = Metal.MTLSize(min(cols, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (cols,))


def gpu_element_sub(a, b):
    """Element-wise subtraction: a - b"""
    a = np.ascontiguousarray(a, dtype=np.float32)
    b = np.ascontiguousarray(b, dtype=np.float32)
    
    total = a.size
    shape = a.shape
    
    a_buffer = create_buffer(a.flatten())
    b_buffer = create_buffer(b.flatten())
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_element_sub)
    encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 2)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, shape)


def gpu_softmax_backward_4d(attW, dattW):
    """Softmax backward: d_scores = attW * (dattW - sum(dattW * attW, axis=-1))"""
    attW = np.ascontiguousarray(attW, dtype=np.float32)
    dattW = np.ascontiguousarray(dattW, dtype=np.float32)
    
    d0, d1, d2, d3 = attW.shape
    total = d0 * d1 * d2 * d3
    
    attW_buffer = create_buffer(attW)
    dattW_buffer = create_buffer(dattW)
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_softmax_backward_4d)
    encoder.setBuffer_offset_atIndex_(attW_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(dattW_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d0), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d1), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d2), 0, 5)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d3), 0, 6)
    
    threads_per_grid = Metal.MTLSize(d3, d2, d0 * d1)
    threads_per_group = Metal.MTLSize(min(d3, 8), min(d2, 8), min(d0 * d1, 4))
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (d0, d1, d2, d3))


def gpu_cross_entropy_loss(probs, labels):
    """Cross entropy loss"""
    probs = np.ascontiguousarray(probs, dtype=np.float32)
    labels = np.ascontiguousarray(labels, dtype=np.uint32)
    
    batch_size, num_classes = probs.shape
    
    probs_buffer = create_buffer(probs)
    labels_buffer = device.newBufferWithBytes_length_options_(labels.tobytes(), labels.nbytes, 0)
    losses_buffer = device.newBufferWithLength_options_(batch_size * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_cross_entropy)
    encoder.setBuffer_offset_atIndex_(probs_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(labels_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(losses_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(batch_size), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(num_classes), 0, 4)
    
    threads_per_grid = Metal.MTLSize(batch_size, 1, 1)
    threads_per_group = Metal.MTLSize(min(batch_size, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    losses = read_buffer(losses_buffer, (batch_size,))
    return float(np.mean(losses))


def gpu_softmax_ce_backward(probs, labels, batch_size):
    """Softmax + cross entropy backward combined"""
    probs = np.ascontiguousarray(probs, dtype=np.float32)
    labels = np.ascontiguousarray(labels, dtype=np.uint32)
    
    bs, num_classes = probs.shape
    scale = 1.0 / batch_size
    
    probs_buffer = create_buffer(probs)
    labels_buffer = device.newBufferWithBytes_length_options_(labels.tobytes(), labels.nbytes, 0)
    dlogits_buffer = device.newBufferWithLength_options_(bs * num_classes * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_softmax_ce_backward)
    encoder.setBuffer_offset_atIndex_(probs_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(labels_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(dlogits_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(bs), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(num_classes), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(scale), 0, 5)
    
    threads_per_grid = Metal.MTLSize(num_classes, bs, 1)
    threads_per_group = Metal.MTLSize(min(num_classes, 16), min(bs, 16), 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(dlogits_buffer, (bs, num_classes))


def gpu_scatter_add(output_shape, indices, updates):
    """Scatter add for embedding gradients"""
    indices = np.ascontiguousarray(indices, dtype=np.uint32).flatten()
    updates = np.ascontiguousarray(updates, dtype=np.float32)
    
    vocab_size, embed_dim = output_shape
    num_indices = indices.shape[0]
    
    # Initialize output to zeros
    output = np.zeros(output_shape, dtype=np.float32)
    out_buffer = create_buffer(output)
    indices_buffer = device.newBufferWithBytes_length_options_(indices.tobytes(), indices.nbytes, 0)
    updates_buffer = create_buffer(updates.reshape(-1, embed_dim))
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_scatter_add)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(indices_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(updates_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(num_indices), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(embed_dim), 0, 4)
    
    threads_per_grid = Metal.MTLSize(embed_dim, num_indices, 1)
    threads_per_group = Metal.MTLSize(min(embed_dim, 64), min(num_indices, 4), 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, output_shape)


def gpu_adam_update(param, grad, m, v, lr, beta1, beta2, epsilon, t):
    """Adam optimizer update - modifies param, m, v in place (returns new values)"""
    param = np.ascontiguousarray(param, dtype=np.float32)
    grad = np.ascontiguousarray(grad, dtype=np.float32)
    m = np.ascontiguousarray(m, dtype=np.float32)
    v = np.ascontiguousarray(v, dtype=np.float32)
    
    total = param.size
    shape = param.shape
    
    beta1_t = beta1 ** t
    beta2_t = beta2 ** t
    
    param_buffer = create_buffer(param.flatten())
    grad_buffer = create_buffer(grad.flatten())
    m_buffer = create_buffer(m.flatten())
    v_buffer = create_buffer(v.flatten())
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_adam_update)
    encoder.setBuffer_offset_atIndex_(param_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(grad_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(m_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(v_buffer, 0, 3)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(lr), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(beta1), 0, 5)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(beta2), 0, 6)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(epsilon), 0, 7)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(beta1_t), 0, 8)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(beta2_t), 0, 9)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    new_param = read_buffer(param_buffer, shape)
    new_m = read_buffer(m_buffer, shape)
    new_v = read_buffer(v_buffer, shape)
    
    return new_param, new_m, new_v

def create_buffer(data):
    """Create a Metal buffer from numpy array"""
    data = np.ascontiguousarray(data, dtype=np.float32)
    return device.newBufferWithBytes_length_options_(data.tobytes(), data.nbytes, 0)

def create_uint_buffer(value):
    """Create a buffer for a single uint32 value"""
    return device.newBufferWithBytes_length_options_(
        np.array([value], dtype=np.uint32).tobytes(), 4, 0)

def create_float_buffer(value):
    """Create a buffer for a single float32 value"""
    return device.newBufferWithBytes_length_options_(
        np.array([value], dtype=np.float32).tobytes(), 4, 0)

def read_buffer(buffer, shape, dtype=np.float32):
    """Read data from Metal buffer into numpy array"""
    size = int(np.prod(shape)) * 4  # 4 bytes per float32
    result = np.frombuffer(
        buffer.contents().as_buffer(size), dtype=dtype
    ).copy()
    return result.reshape(shape)


# ============================================
# GPU Operations
# ============================================

def gpu_dot(a, b):
    """Matrix multiplication: C = A @ B"""
    a = np.ascontiguousarray(a, dtype=np.float32)
    b = np.ascontiguousarray(b, dtype=np.float32)
    
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, f"Dimension mismatch: {K} vs {K2}"
    
    a_buffer = create_buffer(a)
    b_buffer = create_buffer(b)
    result_buffer = device.newBufferWithLength_options_(M * N * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_dot)
    encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(result_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(M), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(K), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(N), 0, 5)
    
    threads_per_grid = Metal.MTLSize(N, M, 1)
    threads_per_group = Metal.MTLSize(min(N, 16), min(M, 16), 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(result_buffer, (M, N))


def gpu_transpose_2d(input_arr):
    """Transpose a 2D array: (M, N) -> (N, M)"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    M, N = input_arr.shape
    
    in_buffer = create_buffer(input_arr)
    out_buffer = device.newBufferWithLength_options_(M * N * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_transpose_2d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(M), 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(N), 0, 3)
    
    threads_per_grid = Metal.MTLSize(N, M, 1)
    threads_per_group = Metal.MTLSize(min(N, 16), min(M, 16), 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (N, M))


def gpu_transpose_4d(input_arr, swap_dims):
    """
    Transpose a 4D array by swapping two dimensions.
    
    Args:
        input_arr: 4D numpy array of shape (d0, d1, d2, d3)
        swap_dims: tuple (dim_a, dim_b) - which dimensions to swap
                   (1, 2) -> swap dim1 and dim2: (d0, d1, d2, d3) -> (d0, d2, d1, d3)
                   (2, 3) -> swap dim2 and dim3: (d0, d1, d2, d3) -> (d0, d1, d3, d2)
    """
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    d0, d1, d2, d3 = input_arr.shape
    total = d0 * d1 * d2 * d3
    
    # Determine mode and output shape
    if swap_dims == (1, 2) or swap_dims == (2, 1):
        mode = 0
        out_shape = (d0, d2, d1, d3)
    elif swap_dims == (2, 3) or swap_dims == (3, 2):
        mode = 1
        out_shape = (d0, d1, d3, d2)
    else:
        raise ValueError(f"Unsupported swap_dims: {swap_dims}. Use (1,2) or (2,3)")
    
    in_buffer = create_buffer(input_arr)
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_transpose_4d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d0), 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d1), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d2), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(d3), 0, 5)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(mode), 0, 6)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, out_shape)


def gpu_layer_norm_forward(x, gamma, beta, eps=1e-5):
    """
    Layer normalization forward pass.
    
    Args:
        x: Input array of shape (batch, seq, embed_dim) or (batch*seq, embed_dim)
        gamma: Scale parameter of shape (embed_dim,)
        beta: Shift parameter of shape (embed_dim,)
        eps: Small constant for numerical stability
    
    Returns:
        output: Normalized output of same shape as x
        cache: (x_norm, var) for backward pass
    """
    original_shape = x.shape
    if len(x.shape) == 3:
        batch, seq, embed_dim = x.shape
        x = x.reshape(batch * seq, embed_dim)
    
    x = np.ascontiguousarray(x, dtype=np.float32)
    gamma = np.ascontiguousarray(gamma, dtype=np.float32)
    beta = np.ascontiguousarray(beta, dtype=np.float32)
    
    rows, cols = x.shape
    
    # Create buffers
    x_buffer = create_buffer(x)
    gamma_buffer = create_buffer(gamma)
    beta_buffer = create_buffer(beta)
    output_buffer = device.newBufferWithLength_options_(rows * cols * 4, 0)
    x_norm_buffer = device.newBufferWithLength_options_(rows * cols * 4, 0)
    var_buffer = device.newBufferWithLength_options_(rows * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_layer_norm_forward)
    encoder.setBuffer_offset_atIndex_(x_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(gamma_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(beta_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(output_buffer, 0, 3)
    encoder.setBuffer_offset_atIndex_(x_norm_buffer, 0, 4)
    encoder.setBuffer_offset_atIndex_(var_buffer, 0, 5)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(rows), 0, 6)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(cols), 0, 7)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(eps), 0, 8)
    
    threads_per_grid = Metal.MTLSize(rows, 1, 1)
    threads_per_group = Metal.MTLSize(min(rows, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    output = read_buffer(output_buffer, (rows, cols))
    x_norm = read_buffer(x_norm_buffer, (rows, cols))
    var = read_buffer(var_buffer, (rows,))
    
    # Reshape back to original shape
    if len(original_shape) == 3:
        output = output.reshape(original_shape)
        x_norm = x_norm.reshape(original_shape)
    
    cache = (x_norm, var, gamma)
    return output, cache


def gpu_layer_norm_backward(dout, cache, eps=1e-5):
    """
    Layer normalization backward pass.
    
    Args:
        dout: Gradient from upstream of same shape as layer norm output
        cache: (x_norm, var, gamma) from forward pass
        eps: Small constant for numerical stability
    
    Returns:
        dx: Gradient w.r.t. input
        dgamma: Gradient w.r.t. gamma
        dbeta: Gradient w.r.t. beta
    """
    x_norm, var, gamma = cache
    
    original_shape = dout.shape
    if len(dout.shape) == 3:
        batch, seq, embed_dim = dout.shape
        dout = dout.reshape(batch * seq, embed_dim)
        x_norm = x_norm.reshape(batch * seq, embed_dim)
    
    dout = np.ascontiguousarray(dout, dtype=np.float32)
    x_norm = np.ascontiguousarray(x_norm, dtype=np.float32)
    var = np.ascontiguousarray(var, dtype=np.float32)
    gamma = np.ascontiguousarray(gamma, dtype=np.float32)
    
    rows, cols = dout.shape
    
    # Create buffers
    dout_buffer = create_buffer(dout)
    x_norm_buffer = create_buffer(x_norm)
    var_buffer = create_buffer(var)
    gamma_buffer = create_buffer(gamma)
    dx_buffer = device.newBufferWithLength_options_(rows * cols * 4, 0)
    dgamma_buffer = device.newBufferWithLength_options_(cols * 4, 0)
    dbeta_buffer = device.newBufferWithLength_options_(cols * 4, 0)
    
    # First kernel: compute dx
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_layer_norm_backward_dx)
    encoder.setBuffer_offset_atIndex_(dout_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(x_norm_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(var_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(gamma_buffer, 0, 3)
    encoder.setBuffer_offset_atIndex_(dx_buffer, 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(rows), 0, 5)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(cols), 0, 6)
    encoder.setBuffer_offset_atIndex_(create_float_buffer(eps), 0, 7)
    
    threads_per_grid = Metal.MTLSize(rows, 1, 1)
    threads_per_group = Metal.MTLSize(min(rows, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    encoder.endEncoding()
    
    # Second kernel: compute dgamma and dbeta
    encoder2 = command_buffer.computeCommandEncoder()
    encoder2.setComputePipelineState_(pipeline_layer_norm_backward_gamma_beta)
    encoder2.setBuffer_offset_atIndex_(dout_buffer, 0, 0)
    encoder2.setBuffer_offset_atIndex_(x_norm_buffer, 0, 1)
    encoder2.setBuffer_offset_atIndex_(dgamma_buffer, 0, 2)
    encoder2.setBuffer_offset_atIndex_(dbeta_buffer, 0, 3)
    encoder2.setBuffer_offset_atIndex_(create_uint_buffer(rows), 0, 4)
    encoder2.setBuffer_offset_atIndex_(create_uint_buffer(cols), 0, 5)
    
    threads_per_grid2 = Metal.MTLSize(cols, 1, 1)
    threads_per_group2 = Metal.MTLSize(min(cols, 256), 1, 1)
    encoder2.dispatchThreads_threadsPerThreadgroup_(threads_per_grid2, threads_per_group2)
    encoder2.endEncoding()
    
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    dx = read_buffer(dx_buffer, (rows, cols))
    dgamma = read_buffer(dgamma_buffer, (cols,))
    dbeta = read_buffer(dbeta_buffer, (cols,))
    
    # Reshape back to original shape
    if len(original_shape) == 3:
        dx = dx.reshape(original_shape)
    
    return dx, dgamma, dbeta


def gpu_softmax(input_arr):
    """2D softmax: softmax applied to each row independently"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    rows, cols = input_arr.shape
    
    in_buffer = create_buffer(input_arr)
    out_buffer = device.newBufferWithLength_options_(rows * cols * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_softmax)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(rows), 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(cols), 0, 3)
    
    threads_per_grid = Metal.MTLSize(rows, 1, 1)
    threads_per_group = Metal.MTLSize(min(rows, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (rows, cols))


def gpu_softmax_4d(input_arr):
    """4D softmax for attention: (batch, heads, seq1, seq2) -> softmax over seq2"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    batch_size, num_heads, seq1, seq2 = input_arr.shape
    total = batch_size * num_heads * seq1 * seq2
    
    in_buffer = create_buffer(input_arr)
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_softmax_4d)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(batch_size), 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(num_heads), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(seq1), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(seq2), 0, 5)
    
    threads_per_grid = Metal.MTLSize(batch_size, num_heads, seq1)
    threads_per_group = Metal.MTLSize(
        min(batch_size, 4), min(num_heads, 4), min(seq1, 16))
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (batch_size, num_heads, seq1, seq2))


def gpu_relu(input_arr):
    """ReLU activation"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    total = input_arr.size
    shape = input_arr.shape
    
    in_buffer = create_buffer(input_arr.flatten())
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_relu)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, shape)


def gpu_relu_backward(input_arr, grad_out):
    """ReLU backward pass"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    grad_out = np.ascontiguousarray(grad_out, dtype=np.float32)
    total = input_arr.size
    shape = input_arr.shape
    
    in_buffer = create_buffer(input_arr.flatten())
    grad_out_buffer = create_buffer(grad_out.flatten())
    grad_in_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_relu_backward)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(grad_out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(grad_in_buffer, 0, 2)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(grad_in_buffer, shape)


def gpu_gelu(input_arr):
    """GELU activation"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    total = input_arr.size
    shape = input_arr.shape
    
    in_buffer = create_buffer(input_arr.flatten())
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_gelu)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 1)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, shape)


def gpu_gelu_backward(input_arr, grad_out):
    """GELU backward pass"""
    input_arr = np.ascontiguousarray(input_arr, dtype=np.float32)
    grad_out = np.ascontiguousarray(grad_out, dtype=np.float32)
    total = input_arr.size
    shape = input_arr.shape
    
    in_buffer = create_buffer(input_arr.flatten())
    grad_out_buffer = create_buffer(grad_out.flatten())
    grad_in_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_gelu_backward)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(grad_out_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(grad_in_buffer, 0, 2)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(grad_in_buffer, shape)


def gpu_element_add(a, b):
    """Element-wise addition"""
    a = np.ascontiguousarray(a, dtype=np.float32)
    b = np.ascontiguousarray(b, dtype=np.float32)
    assert a.shape == b.shape, f"Shape mismatch: {a.shape} vs {b.shape}"
    
    total = a.size
    shape = a.shape
    
    a_buffer = create_buffer(a.flatten())
    b_buffer = create_buffer(b.flatten())
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_element_add)
    encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 2)
    
    threads_per_grid = Metal.MTLSize(total, 1, 1)
    threads_per_group = Metal.MTLSize(min(total, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, shape)


def gpu_embedding_lookup(embed_table, indices):
    """Embedding lookup: output[i,j] = embed_table[indices[i,j]]"""
    embed_table = np.ascontiguousarray(embed_table, dtype=np.float32)
    indices = np.ascontiguousarray(indices, dtype=np.uint32)
    
    vocab_size, embed_dim = embed_table.shape
    
    if len(indices.shape) == 1:
        num_positions = indices.shape[0]
        output_shape = (num_positions, embed_dim)
    else:
        batch_size, seq_len = indices.shape
        num_positions = batch_size * seq_len
        output_shape = (batch_size, seq_len, embed_dim)
    
    embed_buffer = create_buffer(embed_table)
    indices_buffer = device.newBufferWithBytes_length_options_(
        indices.flatten().tobytes(), indices.size * 4, 0)
    output_buffer = device.newBufferWithLength_options_(num_positions * embed_dim * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_embedding)
    encoder.setBuffer_offset_atIndex_(embed_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(indices_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(output_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(num_positions), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(embed_dim), 0, 4)
    
    threads_per_grid = Metal.MTLSize(embed_dim, num_positions, 1)
    threads_per_group = Metal.MTLSize(min(embed_dim, 64), min(num_positions, 4), 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(output_buffer, output_shape)


def gpu_batched_matmul(A, B):
    """
    Batched matrix multiplication for attention.
    A: (batch, heads, M, K)
    B: (batch, heads, K, N)
    Output: (batch, heads, M, N)
    """
    A = np.ascontiguousarray(A, dtype=np.float32)
    B = np.ascontiguousarray(B, dtype=np.float32)
    
    batch_size, num_heads, M, K = A.shape
    _, _, K2, N = B.shape
    assert K == K2, f"Dimension mismatch: {K} vs {K2}"
    
    total = batch_size * num_heads * M * N
    
    a_buffer = create_buffer(A)
    b_buffer = create_buffer(B)
    out_buffer = device.newBufferWithLength_options_(total * 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_batched_matmul)
    encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(out_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(batch_size), 0, 3)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(num_heads), 0, 4)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(M), 0, 5)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(K), 0, 6)
    encoder.setBuffer_offset_atIndex_(create_uint_buffer(N), 0, 7)
    
    threads_per_grid = Metal.MTLSize(N, M, batch_size * num_heads)
    threads_per_group = Metal.MTLSize(min(N, 8), min(M, 8), min(batch_size * num_heads, 4))
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    return read_buffer(out_buffer, (batch_size, num_heads, M, N))


def gpu_dot_3d(a, b):
    """3D matrix multiplication: (batch, seq, K) @ (K, N) -> (batch, seq, N)"""
    batch, seq, K = a.shape
    K2, N = b.shape
    
    a_flat = a.reshape(batch * seq, K)
    result_flat = gpu_dot(a_flat, b)
    return result_flat.reshape(batch, seq, N)


def positional_encoding(block_size, embed_dim):
    pos_enc = np.zeros((block_size, embed_dim), dtype=np.float32)
    for pos in range(block_size):
        for i in range(0, embed_dim, 2):
            pos_enc[pos, i] = np.sin(pos / (10000 ** (i / embed_dim)))
            pos_enc[pos, i + 1] = np.cos(pos / (10000 ** (i / embed_dim)))
    return pos_enc


def cross_entropy_loss(probs, y):
    batch_size = len(y)
    correct_probs = probs[np.arange(batch_size), y]
    return -np.mean(np.log(correct_probs + 1e-10))


def clip_gradient(grad, max_norm=1.0):
    norm = np.sqrt(np.sum(grad ** 2))
    if norm > max_norm:
        return grad * max_norm / norm
    return grad



    
    





    

Metal works! GPU: Apple M4
✓ Shader compiled!
✓ All pipelines created!


: 

In [ ]:
#gpu runner up

n_blocks = 4
epochs = 5
block_size = 8
embed_dim = 64
hidden_size = 512
batch_size = 32
num_heads = 8
head_dim = embed_dim // num_heads
num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size
lr = 0.0001
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
t = 0

# Weight initialization


# Shared: embedding and output
W_embed = np.random.randn(vocab_size, embed_dim).astype(np.float32) * 0.01
W_out = np.random.randn(embed_dim, vocab_size).astype(np.float32) * 0.01
b_out = np.zeros(vocab_size, dtype=np.float32)

m_W_embed, v_W_embed = np.zeros_like(W_embed), np.zeros_like(W_embed)
m_W_out, v_W_out = np.zeros_like(W_out), np.zeros_like(W_out)
m_b_out, v_b_out = np.zeros_like(b_out), np.zeros_like(b_out)

# Per-block weights: each block gets its own set
block_weights = []
block_adam = []

for _ in range(n_blocks):
    w = {
        'ln1_gamma': np.ones(embed_dim, dtype=np.float32),
        'ln1_beta': np.zeros(embed_dim, dtype=np.float32),
        'W_q': np.random.randn(embed_dim, embed_dim).astype(np.float32) * 0.02,
        'W_k': np.random.randn(embed_dim, embed_dim).astype(np.float32) * 0.02,
        'W_v': np.random.randn(embed_dim, embed_dim).astype(np.float32) * 0.02,
        'W_o': np.random.randn(embed_dim, embed_dim).astype(np.float32) * 0.02,
        'ln2_gamma': np.ones(embed_dim, dtype=np.float32),
        'ln2_beta': np.zeros(embed_dim, dtype=np.float32),
        'W_ff1': np.random.randn(embed_dim, hidden_size).astype(np.float32) * np.sqrt(2 / embed_dim),
        'b_ff1': np.zeros(hidden_size, dtype=np.float32),
        'W_ff2': np.random.randn(hidden_size, embed_dim).astype(np.float32) * np.sqrt(2 / hidden_size),
        'b_ff2': np.zeros(embed_dim, dtype=np.float32),
    }
    block_weights.append(w)
    
    # Adam states for this block (m and v for each weight)
    adam = {}
    for key in w:
        adam['m_' + key] = np.zeros_like(w[key])
        adam['v_' + key] = np.zeros_like(w[key])
    block_adam.append(adam)

# Causal mask
CAUSAL_MASK = np.triu(np.ones((block_size, block_size), dtype=np.float32), k=1) * -1e5

PE = positional_encoding(block_size, embed_dim)
PE_batch = np.broadcast_to(PE, (batch_size, block_size, embed_dim)).copy()

# ============================================
# Training Loop (GPU Accelerated)
# ============================================

for epoch in range(epochs):
    indices = np.random.permutation(num_samples)
    X_train_sh = X_train[indices]
    y_train_sh = y_train[indices]

    for batchidx in range(steps_per_epoch):
        start = batchidx * batch_size
        end = start + batch_size
        X_batch = X_train_sh[start:end].astype(np.uint32)
        y_batch = y_train_sh[start:end].astype(np.uint32)

        # ==================== FORWARD PASS ====================
        
        # Embedding + positional encoding
        # Embedding + positional encoding
        embedded = gpu_embedding_lookup(W_embed, X_batch)
        embedded = gpu_element_add(embedded, np.broadcast_to(PE, embedded.shape).copy())

        # Forward through all blocks
        x = embedded
        block_caches = []

        for blk_idx in range(n_blocks):
            w = block_weights[blk_idx]

            # LayerNorm 1 (pre-norm)
            ln1_out, ln1_cache = gpu_layer_norm_forward(x, w['ln1_gamma'], w['ln1_beta'])

            # Attention: Q, K, V projections (on normed input)
            Q = gpu_dot_3d(ln1_out, w['W_q']).reshape(batch_size, block_size, num_heads, head_dim)
            Q = gpu_transpose_4d(Q, (1, 2))
            
            K = gpu_dot_3d(ln1_out, w['W_k']).reshape(batch_size, block_size, num_heads, head_dim)
            K = gpu_transpose_4d(K, (1, 2))
            
            V = gpu_dot_3d(ln1_out, w['W_v']).reshape(batch_size, block_size, num_heads, head_dim)
            V = gpu_transpose_4d(V, (1, 2))

            # Attention scores
            K_T = gpu_transpose_4d(K, (2, 3))
            scores = gpu_batched_matmul(Q, K_T)
            scores = gpu_scalar_div(scores, np.sqrt(head_dim))
            scores = gpu_element_add(scores, np.broadcast_to(CAUSAL_MASK, scores.shape).copy())
            
            attW = gpu_softmax_4d(scores)

            # Attention output
            attOut = gpu_batched_matmul(attW, V)
            attOut_reshaped = gpu_transpose_4d(attOut, (1, 2)).reshape(batch_size, block_size, embed_dim)
            projOut = gpu_dot_3d(attOut_reshaped, w['W_o'])

            # Residual ADD
            res1 = gpu_element_add(x, projOut)

            # LayerNorm 2 (pre-norm)
            ln2_out, ln2_cache = gpu_layer_norm_forward(res1, w['ln2_gamma'], w['ln2_beta'])

            # FFN (on normed input)
            ffn_hidden = gpu_dot_3d(ln2_out, w['W_ff1'])
            ffn_hidden = gpu_bias_add_3d(ffn_hidden, w['b_ff1'])
            ffn_gelu = gpu_gelu(ffn_hidden)
            ffn_out = gpu_dot_3d(ffn_gelu, w['W_ff2'])
            ffn_out = gpu_bias_add_3d(ffn_out, w['b_ff2'])

            # Residual ADD
            res2 = gpu_element_add(res1, ffn_out)

            # Save everything needed for backward
            block_caches.append({
                'x_input': x,
                'ln1_out': ln1_out, 'ln1_cache': ln1_cache,
                'Q': Q, 'K': K, 'V': V, 'attW': attW,
                'attOut_reshaped': attOut_reshaped,
                'res1': res1,
                'ln2_out': ln2_out, 'ln2_cache': ln2_cache,
                'ffn_hidden': ffn_hidden, 'ffn_gelu': ffn_gelu,
            })

            x = res2

        # Output projection (after all blocks)
        x_flat = x.reshape(batch_size * block_size, embed_dim)
        logits = gpu_dot(x_flat, W_out)
        logits = gpu_bias_add_2d(logits, b_out)  # GPU
        prob_softmax = gpu_softmax(logits)

        # Loss
        y_flat = y_batch.flatten().astype(np.uint32)
        loss = gpu_cross_entropy_loss(prob_softmax, y_flat)  # GPU

        # ==================== BACKWARD PASS ====================
        
        
        num_predictions = batch_size * block_size
        dlogits = gpu_softmax_ce_backward(prob_softmax, y_flat, num_predictions)

        
        dW_out = gpu_dot(gpu_transpose_2d(x_flat), dlogits)
        db_out = gpu_sum_2d_to_1d(dlogits)
        dx_flat = gpu_dot(dlogits, gpu_transpose_2d(W_out))

        # Reshape
        dx = dx_flat.reshape(batch_size, block_size, embed_dim)

        # Backward through all blocks (reverse order)
        block_grads = []

        for blk_idx in reversed(range(n_blocks)):
            w = block_weights[blk_idx]
            cache = block_caches[blk_idx]

            # --- FFN + residual backward ---
            dffn_out = dx
            dres1_highway = dx  # gradient highway

            dW_ff2 = gpu_dot(gpu_transpose_2d(cache['ffn_gelu'].reshape(-1, hidden_size)), dffn_out.reshape(-1, embed_dim))
            db_ff2 = gpu_sum_3d_to_1d(dffn_out)
            dffn_gelu = gpu_dot_3d(dffn_out, gpu_transpose_2d(w['W_ff2']))

            dffn_hidden = gpu_gelu_backward(cache['ffn_hidden'], dffn_gelu)

            dW_ff1 = gpu_dot(gpu_transpose_2d(cache['ln2_out'].reshape(-1, embed_dim)), dffn_hidden.reshape(-1, hidden_size))
            db_ff1 = gpu_sum_3d_to_1d(dffn_hidden)
            dln2_out = gpu_dot_3d(dffn_hidden, gpu_transpose_2d(w['W_ff1']))

            dres1_from_ln2, dln2_gamma, dln2_beta = gpu_layer_norm_backward(dln2_out, cache['ln2_cache'])

            dres1 = gpu_element_add(dres1_highway, dres1_from_ln2)

            # --- Attention + residual backward ---
            dprojOut = dres1
            dx_highway = dres1  # gradient highway

            dW_o = gpu_dot(gpu_transpose_2d(cache['attOut_reshaped'].reshape(-1, embed_dim)), dprojOut.reshape(-1, embed_dim))
            d_attOut_combined = gpu_dot_3d(dprojOut, gpu_transpose_2d(w['W_o']))
            d_attOut = d_attOut_combined.reshape(batch_size, block_size, num_heads, head_dim)
            d_attOut = gpu_transpose_4d(d_attOut, (1, 2))

            V_T = gpu_transpose_4d(cache['V'], (2, 3))
            dattW = gpu_batched_matmul(d_attOut, V_T)
            attW_T = gpu_transpose_4d(cache['attW'], (2, 3))
            dV_heads = gpu_batched_matmul(attW_T, d_attOut)

            d_scores = gpu_softmax_backward_4d(cache['attW'], dattW)
            d_scores = gpu_scalar_div(d_scores, np.sqrt(head_dim))

            dQ_heads = gpu_batched_matmul(d_scores, cache['K'])
            d_scores_T = gpu_transpose_4d(d_scores, (2, 3))
            dK_heads = gpu_batched_matmul(d_scores_T, cache['Q'])

            dQ_res = gpu_transpose_4d(dQ_heads, (1, 2)).reshape(batch_size, block_size, embed_dim)
            dK_res = gpu_transpose_4d(dK_heads, (1, 2)).reshape(batch_size, block_size, embed_dim)
            dV_res = gpu_transpose_4d(dV_heads, (1, 2)).reshape(batch_size, block_size, embed_dim)

            dW_q = gpu_dot(gpu_transpose_2d(cache['ln1_out'].reshape(-1, embed_dim)), dQ_res.reshape(-1, embed_dim))
            dW_k = gpu_dot(gpu_transpose_2d(cache['ln1_out'].reshape(-1, embed_dim)), dK_res.reshape(-1, embed_dim))
            dW_v = gpu_dot(gpu_transpose_2d(cache['ln1_out'].reshape(-1, embed_dim)), dV_res.reshape(-1, embed_dim))

            dln1_out = gpu_dot_3d(dQ_res, gpu_transpose_2d(w['W_q']))
            dln1_out = gpu_element_add(dln1_out, gpu_dot_3d(dK_res, gpu_transpose_2d(w['W_k'])))
            dln1_out = gpu_element_add(dln1_out, gpu_dot_3d(dV_res, gpu_transpose_2d(w['W_v'])))

            dx_from_ln1, dln1_gamma, dln1_beta = gpu_layer_norm_backward(dln1_out, cache['ln1_cache'])

            dx = gpu_element_add(dx_highway, dx_from_ln1)

            # Store gradients for this block
            block_grads.append({
                'dW_q': dW_q, 'dW_k': dW_k, 'dW_v': dW_v, 'dW_o': dW_o,
                'dln1_gamma': dln1_gamma, 'dln1_beta': dln1_beta,
                'dW_ff1': dW_ff1, 'db_ff1': db_ff1,
                'dW_ff2': dW_ff2, 'db_ff2': db_ff2,
                'dln2_gamma': dln2_gamma, 'dln2_beta': dln2_beta,
            })

        # dx is now the gradient at the embedding level
        d_embedded = dx
        dW_embed = gpu_scatter_add(W_embed.shape, X_batch, d_embedded)  # GPU

        dW_embed = clip_gradient(dW_embed)
        dW_out = clip_gradient(dW_out)
        db_out = clip_gradient(db_out)

        block_grads.reverse()
        for blk_idx in range(n_blocks):
            g = block_grads[blk_idx]
            for dkey in g:
                g[dkey] = clip_gradient(g[dkey])
        # ==================== ADAM UPDATE ====================
        t += 1
        
        # Embedding and output (shared)
        W_embed, m_W_embed, v_W_embed = gpu_adam_update(W_embed, dW_embed, m_W_embed, v_W_embed, lr, beta1, beta2, epsilon, t)
        W_out, m_W_out, v_W_out = gpu_adam_update(W_out, dW_out, m_W_out, v_W_out, lr, beta1, beta2, epsilon, t)
        b_out, m_b_out, v_b_out = gpu_adam_update(b_out, db_out, m_b_out, v_b_out, lr, beta1, beta2, epsilon, t)

        # Per-block updates (block_grads is in reverse order, so flip it)
        
        for blk_idx in range(n_blocks):
            w = block_weights[blk_idx]
            a = block_adam[blk_idx]
            g = block_grads[blk_idx]

            for key in ['W_q', 'W_k', 'W_v', 'W_o', 'ln1_gamma', 'ln1_beta',
                        'W_ff1', 'b_ff1', 'W_ff2', 'b_ff2', 'ln2_gamma', 'ln2_beta']:
                dkey = 'd' + key
                w[key], a['m_' + key], a['v_' + key] = gpu_adam_update(
                    w[key], g[dkey], a['m_' + key], a['v_' + key],
                    lr, beta1, beta2, epsilon, t
                )

        if batchidx % 50 == 0:
            acc = np.mean(np.argmax(prob_softmax, axis=1) == y_flat)
            print(f"Epoch {epoch+1}, Batch {batchidx}, Loss: {loss:.4f}, Acc: {acc:.4f}")
            gc.collect()

Epoch 1, Batch 0, Loss: 10.1462, Acc: 0.0000
Epoch 1, Batch 50, Loss: 7.7773, Acc: 0.0430


In [ ]:
#Mine2

import Metal
import numpy as np

print("Step 1: Creating device...")
device = Metal.MTLCreateSystemDefaultDevice()
print(f"  GPU: {device.name()}")

print("Step 2: Loading shader...")
try:
    with open('kernel_claude_test.metal', 'r') as f:
        shader_source = f.read()
    print("  Shader file loaded")
except FileNotFoundError:
    print("  ERROR: kernel_claude_test.metal not found!")
    raise

print("Step 3: Compiling shader...")
library, error = device.newLibraryWithSource_options_error_(shader_source, None, None)
if library is None:
    print(f"  COMPILATION ERROR: {error}")
    raise SystemExit(1)
print("  Shader compiled successfully")

print("Step 4: Getting functions...")
functions_to_test = [
    "gpu_dot",
    "gpu_embedding_lookup", 
    "gpu_relu",
    "gpu_relu_backward",
    "gpu_softmax",
    "gpu_softmax_4d",
    "gpu_transpose_2d",
    "gpu_transpose_4d",
    "gpu_element_add",
    "gpu_layer_norm_forward",
    "gpu_layer_norm_backward_dx",
    "gpu_layer_norm_backward_gamma_beta",
    "gpu_batched_matmul",
]

for fn_name in functions_to_test:
    fn = library.newFunctionWithName_(fn_name)
    if fn is None:
        print(f"  ERROR: Function '{fn_name}' not found!")
    else:
        print(f"  ✓ {fn_name}")

print("Step 5: Creating pipelines...")
command_queue = device.newCommandQueue()

# Test just gpu_dot
dot_fn = library.newFunctionWithName_("gpu_dot")
pipeline_dot, error = device.newComputePipelineStateWithFunction_error_(dot_fn, None)
if pipeline_dot is None:
    print(f"  Pipeline error: {error}")
else:
    print("  ✓ pipeline_dot created")

print("Step 6: Testing simple matrix multiply...")
def create_buffer(data):
    data = np.ascontiguousarray(data, dtype=np.float32)
    return device.newBufferWithBytes_length_options_(data.tobytes(), data.nbytes, 0)

def create_uint_buffer(value):
    return device.newBufferWithBytes_length_options_(
        np.array([value], dtype=np.uint32).tobytes(), 4, 0)

# Small test
A = np.array([[1, 2], [3, 4]], dtype=np.float32)
B = np.array([[5, 6], [7, 8]], dtype=np.float32)
M, K = 2, 2
N = 2

a_buffer = create_buffer(A)
b_buffer = create_buffer(B)
result_buffer = device.newBufferWithLength_options_(M * N * 4, 0)

command_buffer = command_queue.commandBuffer()
encoder = command_buffer.computeCommandEncoder()
encoder.setComputePipelineState_(pipeline_dot)
encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
encoder.setBuffer_offset_atIndex_(result_buffer, 0, 2)
encoder.setBuffer_offset_atIndex_(create_uint_buffer(M), 0, 3)
encoder.setBuffer_offset_atIndex_(create_uint_buffer(K), 0, 4)
encoder.setBuffer_offset_atIndex_(create_uint_buffer(N), 0, 5)

threads_per_grid = Metal.MTLSize(N, M, 1)
threads_per_group = Metal.MTLSize(2, 2, 1)
encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)

encoder.endEncoding()
command_buffer.commit()
command_buffer.waitUntilCompleted()

result = np.frombuffer(result_buffer.contents().as_buffer(16), dtype=np.float32).copy().reshape(2, 2)
expected = A @ B

print(f"  GPU result:\n{result}")
print(f"  Expected:\n{expected}")
print(f"  Match: {np.allclose(result, expected)}")

print("\n✓ All basic tests passed!")

Step 1: Creating device...
  GPU: Apple M4
Step 2: Loading shader...
  Shader file loaded
Step 3: Compiling shader...
  COMPILATION ERROR: Error Domain=MTLLibraryErrorDomain Code=3 "program_source:593:39: error: unknown type name 'atomic_float'
    atomic_fetch_add_explicit((device atomic_float*)&output[out_idx], 
                                      ^
" UserInfo={NSLocalizedDescription=program_source:593:39: error: unknown type name 'atomic_float'
    atomic_fetch_add_explicit((device atomic_float*)&output[out_idx], 
                                      ^
}


SystemExit: 1

In [ ]:
#MINE

import Metal
import numpy as np


device = Metal.MTLCreateSystemDefaultDevice()
print(f"Metal works! GPU: {device.name()}")

# Load and compile WITH error checking
with open('kernel_sc.metal', 'r') as f:
    shader_source = f.read()

library, error = device.newLibraryWithSource_options_error_(
    shader_source, None, None
)

# Check if compilation failed
if library is None:
    print("❌ SHADER COMPILATION FAILED!")
    print(f"Error: {error}")
    raise SystemExit(1)

print("✓ Shader compiled!")
dot_fn = library.newFunctionWithName_("gpu_dot")
add_fn = library.newFunctionWithName_("gpu_add")
relu_fn = library.newFunctionWithName_("gpu_relu_derivative")  # Fixed typo!
relu_deriv_fn = library.newFunctionWithName_("gpu_relu_derivative")
softmax_fn = library.newFunctionWithName_("gpu_softmax")
exp_fn = library.newFunctionWithName_("gpu_exp")
square_fn = library.newFunctionWithName_("gpu_square")
embedding_lookup_fn = library.newFunctionWithName_("gpu_embedding_lookup")
sqrt_fn = library.newFunctionWithName_("gpu_sqrt")
layer_norm_fn = library.newFunctionWithName_("gpu_layer_norm_apply")

if dot_fn is None:
    print("❌ Function 'gpu_add' not found!")
    raise SystemExit(1)


print("✓ Function found!")

pipeline_add = device.newComputePipelineStateWithFunction_error_(add_fn, None)[0]
pipeline_dot = device.newComputePipelineStateWithFunction_error_(dot_fn, None)[0]
pipeline_embedding = device.newComputePipelineStateWithFunction_error_(embedding_lookup_fn, None)[0]
pipeline_relu = device.newComputePipelineStateWithFunction_error_(relu_fn, None)[0]
pipeline_softmax = device.newComputePipelineStateWithFunction_error_(softmax_fn, None)[0]






command_queue = device.newCommandQueue()

def gpu_softmax(input):
    input = np.ascontiguousarray(input, dtype=np.float32)
    
    rows, cols = input.shape
    
    in_buffer = device.newBufferWithBytes_length_options_(
        input.tobytes(), input.nbytes, 0)
    
    output_size = rows * cols * 4
    output_buffer = device.newBufferWithLength_options_(output_size, 0)
    
    rows_buffer = device.newBufferWithBytes_length_options_(
        np.array([rows], dtype=np.uint32).tobytes(), 4, 0)
    cols_buffer = device.newBufferWithBytes_length_options_(
        np.array([cols], dtype=np.uint32).tobytes(), 4, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    
    encoder.setComputePipelineState_(pipeline_softmax)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(output_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(rows_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(cols_buffer, 0, 3)
    
    threads_per_grid = Metal.MTLSize(rows, 1, 1)
    threads_per_group = Metal.MTLSize(min(rows, 256), 1, 1)
    
    encoder.dispatchThreads_threadsPerThreadgroup_(
        threads_per_grid,
        threads_per_group
    )
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    result = np.frombuffer(
        output_buffer.contents().as_buffer(output_size), 
        dtype=np.float32
    ).copy()
    
    result = result.reshape(rows, cols)
    return result
def kernel_dot(a,b):

    a = np.ascontiguousarray(a, dtype=np.float32)#makes sure not float64 and that memory is all together 
    b = np.ascontiguousarray(b, dtype=np.float32)

    M,K = a.shape #A is (M rows, K columns)
    K2, N = b.shape     # B is (K rows, N columns)

    # Convert dimensions to bytes
    M_bytes = np.array([M], dtype=np.uint32).tobytes()
    K_bytes = np.array([K], dtype=np.uint32).tobytes()
    N_bytes = np.array([N], dtype=np.uint32).tobytes()

    # Create GPU buffers for dimensions
    M_buffer = device.newBufferWithBytes_length_options_(M_bytes, 4, 0)
    K_buffer = device.newBufferWithBytes_length_options_(K_bytes, 4, 0)
    N_buffer = device.newBufferWithBytes_length_options_(N_bytes, 4, 0)

    a_buffer = device.newBufferWithBytes_length_options_(a.tobytes() , a.nbytes,0)
    b_buffer = device.newBufferWithBytes_length_options_(b.tobytes() , b.nbytes,0)
    result_size = M * N *4 #4bytes per float
    result_buffer = device.newBufferWithLength_options_(result_size, 0)

    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline_dot)
    encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(result_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(M_buffer, 0, 3)
    encoder.setBuffer_offset_atIndex_(K_buffer, 0, 4)
    encoder.setBuffer_offset_atIndex_(N_buffer, 0, 5)

    threads_per_grid = Metal.MTLSize(N, M, 1) 
    threads_per_group = Metal.MTLSize(min(N, 16),min(M, 16), 1)

    encoder.dispatchThreads_threadsPerThreadgroup_(
        threads_per_grid,
        threads_per_group
    )
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()

    result = np.frombuffer(
        result_buffer.contents().as_buffer(result_size),
        dtype=np.float32
    ).copy().reshape(M, N)

    return result 

def kernel_relu(input):
    input = np.ascontiguousarray(input , dtype=np.float32)
    M,N = input.shape
    total_elements = M*N
    in_buffer = device.newBufferWithBytes_length_options_(input.tobytes(),input.nbytes,0)
    output_size = M * N * 4
    output_buffer = device.newBufferWithLength_options_(output_size, 0)

    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()

    encoder.setComputePipelineState_(relu_fn)
    encoder.setBuffer_offset_atIndex_(in_buffer,0,0)
    encoder.setBuffer_offset_atIndex_(output_buffer,0,1)
    
    threads_per_grid = Metal.MTLSize(total_elements,1,1)
    threads_per_group = Metal.MTLSize(min(total_elements, 256), 1,1)
    encoder.dispatchThreads_threadsPerThreadgroup_(
        threads_per_grid,
        threads_per_group
    )
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()

    result = np.frombuffer(
        output_buffer.contents().as_buffer(output_size), dtype=np.float32).copy()
    
    result = result
def gpu_softmax(input):
    input = np.ascontiguousarray(input, dtype=np.float32)
    
    rows, cols = input.shape
    
    
    in_buffer = device.newBufferWithBytes_length_options_(
        input.tobytes(), input.nbytes, 0)
    
   
    output_size = rows * cols * 4  # 4 bytes per float
    output_buffer = device.newBufferWithLength_options_(output_size, 0)
    
    # Create dimension buffers
    rows_buffer = device.newBufferWithBytes_length_options_(
        np.array([rows], dtype=np.uint32).tobytes(), 4, 0)
    cols_buffer = device.newBufferWithBytes_length_options_(
        np.array([cols], dtype=np.uint32).tobytes(), 4, 0)
    
    # Create command buffer and encoder
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    
    # Set pipeline and buffers
    encoder.setComputePipelineState_(pipeline_softmax)
    encoder.setBuffer_offset_atIndex_(in_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(output_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(rows_buffer, 0, 2)
    encoder.setBuffer_offset_atIndex_(cols_buffer, 0, 3)
    
    # Dispatch threads (one thread per row)
    threads_per_grid = Metal.MTLSize(rows, 1, 1)
    threads_per_group = Metal.MTLSize(min(rows, 256), 1, 1)
    
    encoder.dispatchThreads_threadsPerThreadgroup_(
        threads_per_grid,
        threads_per_group
    )
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    # Read result
    result = np.frombuffer(
        output_buffer.contents().as_buffer(output_size), 
        dtype=np.float32
    ).copy()
    
    result = result.reshape(rows, cols)
    return result

def embedding_lookup_fn(embed_table,indices):
    embed_table = np.ascontiguousarray(embed_table, dtype=np.float32)
    indices = np.ascontiguousarray(indices, dtype=np.float32)

    vocab_size , embed_dim = embed_table.shape 
    batch_size , seq_len = indices.shape 
    num_pos = batch_size *seq_len

    embed_buffer = device.newBufferWithBytes_length_options_(embed_table.tobytes(),embed_table.nbytes,0)
    indices_flat = indices.flatten()
    indices_buffer = device.newBufferWithBytes_length_options_(indices_flat.tobytes(),indices_flat.nbytes,0)

    output_size = num_pos * embed_dim * 4
    output_buffer = device.newBufferWithLength_options_(output_size, 0)

    num_pos_buffer = device.newBufferWithBytes_length_options_(np.array([num_pos], dtype=np.uint32).tobytes(),4, 0)
    
    embed_dim_buffer = device.newBufferWithBytes_length_options_(np.array([embed_dim], dtype=np.uint32).tobytes(),4,0)



    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()

    encoder.setComputePipelineState_(embedding_lookup_fn)
    encoder.setBuffer_offset_atIndex(embed_buffer,0,0)
    encoder.setBuffer_offset_atIndex(indices_buffer,0,1)
    encoder.setBuffer_offset_atIndex(output_buffer,0,2)
    encoder.setBuffer_offset_atIndex(num_pos_buffer,0,3)
    encoder.setBuffer_offset_atIndex(embed_dim_buffer,0,4)

    threads_per_grid = Metal.MTLSize(embed_dim,num_pos,1)
    threads_per_group = Metal.MTLSize(min(embed_dim, 64), min(num_pos, 4),1)

    print(f"  Launching {embed_dim} × {num_pos} = {embed_dim * num_pos} threads")

    encoder.dispatchThreads_threadsPerThreadgroup_(
        threads_per_grid,
        threads_per_group
    )

    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()

    result = np.frombuffer(
        output_buffer.contents().as_buffer(output_size), dtype=np.float32).copy()
    
    result = result.reshape(batch_size, seq_len, embed_dim)
    






def gpu_add(a, b):
    n = len(a)
    
    a_buffer = device.newBufferWithBytes_length_options_(a.tobytes(), a.nbytes, 0)
    b_buffer = device.newBufferWithBytes_length_options_(b.tobytes(), b.nbytes, 0)
    result_buffer = device.newBufferWithLength_options_(a.nbytes, 0)
    
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    
    encoder.setComputePipelineState_(gpu_add)
    encoder.setBuffer_offset_atIndex_(a_buffer, 0, 0)
    encoder.setBuffer_offset_atIndex_(b_buffer, 0, 1)
    encoder.setBuffer_offset_atIndex_(result_buffer, 0, 2)
    
    threads_per_grid = Metal.MTLSize(n, 1, 1)
    threads_per_group = Metal.MTLSize(min(n, 256), 1, 1)
    encoder.dispatchThreads_threadsPerThreadgroup_(threads_per_grid, threads_per_group)
    
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    
    result = np.frombuffer(
        result_buffer.contents().as_buffer(a.nbytes),
        dtype=np.float32
    ).copy()
    return result


# Test
a = np.array([1, 2, 3, 4, 5], dtype=np.float32) 
b = np.array([5, 4, 3, 2, 1], dtype=np.float32)  
c = gpu_add(a, b)
print(f"Result: {c}")


A = np.array([[1, 2, 3, 4],#shape(3, 4)
     [5, 6, 7, 8], # M=3 K=4
     [9,10,11,12]]
    , dtype=np.float32)

B = np.array([[1, 2],#shape(4, 2)
     [3, 4],#K2=4 N=2
     [5, 6],
     [7, 8]], dtype=np.float32)

C = gpu_sqrt(A)
print(C)

embed_table = np.array([
    [0.1, 0.2, 0.3, 0.4],  # Token 0: "the"
    [0.5, 0.6, 0.7, 0.8],  # Token 1: "cat"
    [0.9, 1.0, 1.1, 1.2],  # Token 2: "sat"
    [1.3, 1.4, 1.5, 1.6],  # Token 3: "on"
    [1.7, 1.8, 1.9, 2.0],  # Token 4: "mat"
], dtype=np.float32)

print(f"\nEmbedding table (vocab_size=5, embed_dim=4):")
print(embed_table)


indices = np.array([
    [1, 2, 3],  # Batch 0: "cat sat on"
    [0, 4, 2],  # Batch 1: "the mat sat"
], dtype=np.uint32)

gpu_result = embedding_lookup_fn(embed_table, indices)

npres = embed_table[indices]

print(f"\nGPU Result (batch_size=2, seq_len=3, embed_dim=4):")
print(gpu_result)

# NumPy comparison
numpy_result = embed_table[indices]
print(f"\nNumPy Result (embed_table[indices]):")
print(numpy_result)

Metal works! GPU: Apple M4


FileNotFoundError: [Errno 2] No such file or directory: 'kernel_sc.metal'

In [ ]:
#MINE
def positional_encoding(block_size, embed_dim):
    pos_enc = np.zeros((block_size, embed_dim))
    for pos in range(block_size):
        for i in range(0,embed_dim,2):
            pos_enc[pos,i] = np.sin(pos / (10000 ** (i / embed_dim)))
            pos_enc[pos,i +1] = np.cos(pos / (10000 **(i/embed_dim)))
    return pos_enc

def softmax(x):
    x_shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, gamma, beta, eps=1e-5):
    """Forward pass for layer normalization"""
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    out = gamma * x_norm + beta
    cache = (x_norm, var, gamma)  # Save for backward
    return out, cache


def layer_norm_backward(dout, cache):
    """Backward pass for layer normalization"""
    x_norm, var, gamma = cache
    
    # Gradients for gamma and beta
    dgamma = np.sum(dout * x_norm, axis=(0, 1))
    dbeta = np.sum(dout, axis=(0, 1))
    
    # Gradient for input
    dx = dout * gamma / np.sqrt(var + 1e-5)
    
    return dx, dgamma, dbeta

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(np.float32)

def cross_entropy_loss(probs, y):
    batch_size = len(y)
    correct_probs = probs[np.arange(batch_size), y]
    return -np.mean(np.log(correct_probs + 1e-10))

def clip_gradient(grad, max_norm=1.0):
    norm = np.sqrt(np.sum(grad ** 2))
    if norm > max_norm:
        return grad * max_norm / norm
    return grad


n_blocks =4
epochs =5 
block_size = 8
embed_dim = 64
hidden_size = 512
epochs = 5
batch_size = 32
num_heads = 8
head_dim = embed_dim // num_heads
input_size = block_size * embed_dim
num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size
lr = 0.0003
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
t = 0
#decay = 1e-4


#layer and vocab weight init 
W_embed = np.random.randn(vocab_size, embed_dim) * 0.01
#W_1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
#W_2 = np.random.randn(hidden_size, hidden_size) * np.sqrt(2 / hidden_size)
#W_3 = np.random.randn(hidden_size, vocab_size) * np.sqrt(2 / hidden_size)
#b1 = np.zeros(hidden_size)
#b2 = np.zeros(hidden_size)
#b3 = np.zeros(vocab_size)


#layer norm 1
ln1_gamma = np.ones(embed_dim)
ln1_beta = np.zeros(embed_dim)

#Weights
W_ff1 = np.random.randn(embed_dim, hidden_size) * np.sqrt(2 / embed_dim)
b_ff1 = np.zeros(hidden_size)
W_ff2 = np.random.randn(hidden_size, embed_dim) * np.sqrt(2 / hidden_size)
b_ff2 = np.zeros(embed_dim)

#attention layer weight init
W_q = np.random.randn(embed_dim, embed_dim) * 0.02
W_k = np.random.randn(embed_dim, embed_dim) * 0.02
W_v = np.random.randn(embed_dim, embed_dim) * 0.02
W_o = np.random.randn(embed_dim, embed_dim) * 0.02

#layer norm 2

ln2_gamma = np.ones(embed_dim)
ln2_beta = np.zeros(embed_dim)

#out projection after tranformer 
W_out = np.random.randn(embed_dim, vocab_size).astype(np.float32) * 0.01
b_out = np.zeros(vocab_size)

#optimizer momentum and velocity weihgt init
m_W_embed, v_W_embed = np.zeros_like(W_embed), np.zeros_like(W_embed)
m_Wq, v_Wq = np.zeros_like(W_q), np.zeros_like(W_q)
m_Wk, v_Wk = np.zeros_like(W_k), np.zeros_like(W_k)
m_Wv, v_Wv = np.zeros_like(W_v), np.zeros_like(W_v)
m_Wo, v_Wo = np.zeros_like(W_o), np.zeros_like(W_o)
m_ln1_gamma, v_ln1_gamma = np.zeros_like(ln1_gamma), np.zeros_like(ln1_gamma)
m_ln1_beta, v_ln1_beta = np.zeros_like(ln1_beta), np.zeros_like(ln1_beta)
m_W_ff1, v_W_ff1 = np.zeros_like(W_ff1), np.zeros_like(W_ff1)
m_b_ff1, v_b_ff1 = np.zeros_like(b_ff1), np.zeros_like(b_ff1)
m_W_ff2, v_W_ff2 = np.zeros_like(W_ff2), np.zeros_like(W_ff2)
m_b_ff2, v_b_ff2 = np.zeros_like(b_ff2), np.zeros_like(b_ff2)
m_ln2_gamma, v_ln2_gamma = np.zeros_like(ln2_gamma), np.zeros_like(ln2_gamma)
m_ln2_beta, v_ln2_beta = np.zeros_like(ln2_beta), np.zeros_like(ln2_beta)
m_W_out, v_W_out = np.zeros_like(W_out), np.zeros_like(W_out)
m_b_out, v_b_out = np.zeros_like(b_out), np.zeros_like(b_out)

#prevents model from looking at future words when predicting token 
CAUSAL_MASK = np.triu(np.ones((block_size, block_size)), k=1) * -1e5 

PE = positional_encoding(block_size,embed_dim)#(8,64)
for epoch in range(epochs) :
    indices = np.random.permutation(num_samples)
    X_train_sh = X_train[indices]
    y_train_sh = y_train[indices]

    for batchidx in range(steps_per_epoch):
        start = batchidx * batch_size
        end = start + batch_size
        X_batch = X_train_sh[start:end]#(32,8) 32 rows 8 columns 
        y_batch = y_train_sh[start:end]#(32,)
        print(y_batch.shape)
        #Forward
        embedded = W_embed[X_batch] + PE
        
        #print(embedded.shape)
        #Attention
        Q = np.dot(embedded, W_q).reshape(batch_size, block_size, num_heads, head_dim).transpose(0, 2, 1, 3) #without reshape (32,8,64)
        #print(Q[0][0][0][0])
        #|print('q',Q[0][0][0][0])
        K = np.dot(embedded, W_k).reshape(batch_size, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)
        V = np.dot(embedded, W_v).reshape(batch_size, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)

        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
        scores = scores + CAUSAL_MASK
        attW = softmax(scores)
        print(attW.shape)

        attOut = np.matmul(attW, V)
        attOut_reshaped = attOut.transpose(0, 2, 1, 3).reshape(batch_size, block_size, embed_dim)
        projOut = np.dot(attOut_reshaped, W_o)

        #layer norm 1 plus residual 
        res1 = embedded + projOut
        ln1_out, ln1_cache = layer_norm(res1, ln1_gamma, ln1_beta)

        #FNN
        ffn_hidden = np.dot(ln1_out, W_ff1) + b_ff1
        ffn_relu = relu(ffn_hidden)
        ffn_out = np.dot(ffn_relu, W_ff2) + b_ff2

        #layer norm 2 plus residual 
        res2 = ln1_out + ffn_out
        ln2_out, ln2_cache = layer_norm(res2, ln2_gamma, ln2_beta)

        #output last token 
        last_hidden = ln2_out[:, -1, :]
        logits = np.dot(last_hidden, W_out) + b_out
        prob_softmax = softmax(logits)

        #Loss
        loss = cross_entropy_loss(prob_softmax, y_batch)
        


        #Back 
        dlogits = prob_softmax.copy()
        dlogits[np.arange(batch_size), y_batch] -= 1
        
        dlogits=dlogits / batch_size
        print("bsdfsadf",dlogits.shape)

        dW_out = np.dot(last_hidden.T, dlogits)
        db_out = np.sum(dlogits, axis=0)
        dlast_hidden = np.dot(dlogits, W_out.T)
        
        #full sequence 
        dln2_out = np.zeros((batch_size, block_size, embed_dim))
        dln2_out[:, -1, :] = dlast_hidden

        #layer norm 2 res back 
        dres2, dln2_gamma, dln2_beta = layer_norm_backward(dln2_out, ln2_cache)

        dffn_out = dres2
        dln1_out_res = dres2

        #layer norm  back 
        dW_ff2 = np.dot(ffn_relu.reshape(-1, hidden_size).T, dffn_out.reshape(-1, embed_dim))
        db_ff2 = np.sum(dffn_out, axis=(0, 1))
        dffn_relu = np.dot(dffn_out, W_ff2.T)

        dffn_hidden = dffn_relu * relu_derivative(ffn_hidden)

        dW_ff1 = np.dot(ln1_out.reshape(-1, embed_dim).T, dffn_hidden.reshape(-1, hidden_size))
        db_ff1 = np.sum(dffn_hidden, axis=(0, 1))
        dln1_out_ffn = np.dot(dffn_hidden, W_ff1.T)

        # Combine gradients to ln1_out
        dln1_out = dln1_out_res + dln1_out_ffn

        # LayerNorm 1 backward
        dres1, dln1_gamma, dln1_beta = layer_norm_backward(dln1_out, ln1_cache)

        # Residual 1 backward
        dprojOut = dres1
        d_embedded_res = dres1

        # Attention backward
        dW_o = np.dot(attOut_reshaped.reshape(-1, embed_dim).T, dprojOut.reshape(-1, embed_dim))
        d_attOut_combined = np.dot(dprojOut, W_o.T)
        d_attOut = d_attOut_combined.reshape(batch_size, block_size, num_heads, head_dim)
        d_attOut = d_attOut.transpose(0, 2, 1, 3)

        dattW = np.matmul(d_attOut, V.transpose(0, 1, 3, 2))
        dV_heads = np.matmul(attW.transpose(0, 1, 3, 2), d_attOut)

        d_scores = attW * (dattW - np.sum(dattW * attW, axis=-1, keepdims=True))
        d_scores = d_scores / np.sqrt(head_dim)

        dQ_heads = np.matmul(d_scores, K)
        dK_heads = np.matmul(d_scores.transpose(0, 1, 3, 2), Q)

        dQ_res = dQ_heads.transpose(0, 2, 1, 3)
        dQ_res = dQ_res.reshape(batch_size, block_size, embed_dim)
        
        dK_res = dK_heads.transpose(0, 2, 1, 3)
        dK_res = dK_res.reshape(batch_size, block_size, embed_dim)
        
        dV_res = dV_heads.transpose(0, 2, 1, 3)
        dV_res = dV_res.reshape(batch_size, block_size, embed_dim)

        dW_q = np.dot(embedded.reshape(-1, embed_dim).T, dQ_res.reshape(-1, embed_dim))
        dW_k = np.dot(embedded.reshape(-1, embed_dim).T, dK_res.reshape(-1, embed_dim))
        dW_v = np.dot(embedded.reshape(-1, embed_dim).T, dV_res.reshape(-1, embed_dim))

        d_embedded = np.dot(dQ_res, W_q.T) + np.dot(dK_res, W_k.T) + np.dot(dV_res, W_v.T)
        d_embedded = d_embedded + d_embedded_res
        
        dW_embed = np.zeros_like(W_embed)
        np.add.at(dW_embed, X_batch, d_embedded)

        # ==================== GRADIENT CLIPPING ====================
        # Clip every gradient to max norm 1.0 before Adam update.
        # This prevents any single large gradient from causing weight
        # explosions that cascade into NaN.
        
        dW_embed = clip_gradient(dW_embed)
        dW_out = clip_gradient(dW_out)
        db_out = clip_gradient(db_out)

        block_grads.reverse()
        for blk_idx in range(n_blocks):
            g = block_grads[blk_idx]
            for dkey in g:
                g[dkey] = clip_gradient(g[dkey])

        # ==================== ADAM UPDATE ====================
        t += 1
        
        # Embedding and output (shared)
        W_embed, m_W_embed, v_W_embed = gpu_adam_update(W_embed, dW_embed, m_W_embed, v_W_embed, lr, beta1, beta2, epsilon, t)
        W_out, m_W_out, v_W_out = gpu_adam_update(W_out, dW_out, m_W_out, v_W_out, lr, beta1, beta2, epsilon, t)
        b_out, m_b_out, v_b_out = gpu_adam_update(b_out, db_out, m_b_out, v_b_out, lr, beta1, beta2, epsilon, t)

        # Per-block updates
        for blk_idx in range(n_blocks):
            w = block_weights[blk_idx]
            a = block_adam[blk_idx]
            g = block_grads[blk_idx]

            for key in ['W_q', 'W_k', 'W_v', 'W_o', 'ln1_gamma', 'ln1_beta',
                        'W_ff1', 'b_ff1', 'W_ff2', 'b_ff2', 'ln2_gamma', 'ln2_beta']:
                dkey = 'd' + key
                w[key], a['m_' + key], a['v_' + key] = gpu_adam_update(
                    w[key], g[dkey], a['m_' + key], a['v_' + key],
                    lr, beta1, beta2, epsilon, t
                )
        if batchidx % 100 == 0:
            acc = np.mean(np.argmax(prob_softmax, axis=1) == y_batch)
            last_attn = attW[0, 0, -1, :]
            print(f"Epoch {epoch+1}, Batch {batchidx}, Loss: {loss:.4f}, Acc: {acc:.4f}, attn: {last_attn.max()}")
            print(f"  Attention: {last_attn.round(3)}")
        

In [ ]:
# After epoch 1, test on validation set:
def evaluate(X_test, y_test):
    correct = 0
    total = 0
    for i in range(0, len(X_test), batch_size):
        X_b = X_test[i:i+batch_size]
        y_b = y_test[i:i+batch_size]
        
        # Forward pass (copy from training)
        embedded = W_embed[X_b] + PE
        Q = np.dot(embedded, W_q).reshape(-1, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)
        K = np.dot(embedded, W_k).reshape(-1, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)
        V = np.dot(embedded, W_v).reshape(-1, block_size, num_heads, head_dim).transpose(0, 2, 1, 3)
        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim) + CAUSAL_MASK
        attW = softmax(scores)
        attOut = np.matmul(attW, V).transpose(0, 2, 1, 3).reshape(-1, block_size, embed_dim)
        projOut = np.dot(attOut, W_o)
        res1 = embedded + projOut
        ln1_out, _ = layer_norm(res1, ln1_gamma, ln1_beta)
        ffn_hidden = relu(np.dot(ln1_out, W_ff1) + b_ff1)
        ffn_out = np.dot(ffn_hidden, W_ff2) + b_ff2
        res2 = ln1_out + ffn_out
        ln2_out, _ = layer_norm(res2, ln2_gamma, ln2_beta)
        logits = np.dot(ln2_out[:, -1, :], W_out) + b_out
        
        preds = np.argmax(logits, axis=1)
        correct += np.sum(preds == y_b)
        total += len(y_b)
    
    return correct / total

# Run after epoch 1
test_acc = evaluate(X_test[:10000], y_test[:10000])  # Sample for speed
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
block_size = 8
embed_dim = 64
hidden_size = 256
epochs = 5
batch_size = 32
num_heads = 4
head_dim = embed_dim // num_heads
input_size = block_size * embed_dim
num_samples = X_train.shape[0]
steps_per_epoch = num_samples // batch_size

